# Perform balanced sampling

In [1]:
import pandas as pd

In [2]:
label_data = pd.read_parquet("../data/processed/chain/labeled_days.parquet")
summary_df = pd.read_csv("../data/processed/chain/labeled_days_summary.csv")

**check address which has more anomaly days than normal days**

In [3]:
summary_df['normal_days'] = summary_df['total_active_days'] - summary_df['anomaly_days']

addresses_anom_gt_norm = summary_df[summary_df['anomaly_days'] > summary_df['normal_days']]

print(f"Number of addresses where anomalous > normal: {len(addresses_anom_gt_norm)}")
addresses_anom_gt_norm[['address', 'normal_days', 'anomaly_days']]


Number of addresses where anomalous > normal: 6


,address,normal_days,anomaly_days
32,0x03d18450b2ef96a94b2930e4c36d8a667ce51fc9,24,27
892,0x54920ae6333e9dfab74a968ebf1b3468ca07ccf5,8,10
1579,0x91af8014a7f74ab1cf1eedb0aa96f58dced95f38,12,14
1607,0x94d50168d9bc80cc39affbcba8232acd89f117a3,22,23
1946,0xb294b24d67a4ee18098e80e9efb9d47394a65993,9,10
2640,0xf4a2eff88a408ff4c4550148151c33c93442619e,158,291


**balancing the dataset**

In [5]:
balanced_samples = []

for address, group in label_data.groupby("address"):
    anomalous = group[group['is_anomalous'] == 1]
    normal = group[group['is_anomalous'] == 0]

    if len(anomalous) > len(normal):
        # Ignore anomalies, keep only normal rows
        balanced_addr_df = normal
    else:
        # Keep all anomalies and sample equal number of normals
        normal_sampled = normal.sample(n=len(anomalous), random_state=42)
        balanced_addr_df = pd.concat([anomalous, normal_sampled])

    balanced_samples.append(balanced_addr_df)

# Combine all addresses
balanced_per_address_df = (
    pd.concat(balanced_samples)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print(f"Total days: {len(balanced_per_address_df)}")
print(balanced_per_address_df['is_anomalous'].value_counts())
print(balanced_per_address_df['is_anomalous'].value_counts(normalize=True) * 100)

# Sort by day
balanced_per_address_df = balanced_per_address_df.sort_values('day').reset_index(drop=True)

output_parquet = "../data/processed/chain/balanced_label_days.parquet"
balanced_per_address_df.to_parquet(output_parquet, index=False)
print(f"Saved to: {output_parquet}")


Total days: 59575
is_anomalous
0    29904
1    29671
Name: count, dtype: int64
is_anomalous
0    50.195552
1    49.804448
Name: proportion, dtype: float64
Saved to: ../data/processed/chain/balanced_label_days.parquet
